# Asynchronous external evaluation with ask/tell in Emukit

## Overview

In some experiments the evaluation of the objective function is started externally and **finishes at an
unpredictable time**: you submit a batch of experiments to a lab queue or a cluster, and each one returns
whenever it completes. In that setting the synchronous
`get_next_points` / `run_loop` workflows are awkward because they assume a result is available immediately
after every suggested point.

Emukit's :class:`~emukit.core.loop.OuterLoop` therefore offers an asynchronous **ask/tell** interface:

- ``ask()`` claims a candidate point and returns it together with a traceable **observation id**. The point
  is registered as *pending* but is **not** treated as a real observation.
- ``tell(result, observation_id)`` confirms a successful experiment. Reports may arrive **out of order**;
  only the first confirmed success for an id is accepted, and conflicting or repeated reports are rejected.
- ``tell_failure`` and ``tell_cancelled`` release the placeholder of an experiment that failed or was
  cancelled without ever adding an observation to the loop state.

Pending points participate in candidate selection (they are excluded or penalized so the same point is not
suggested twice while an experiment is in flight), but they never enter the
:class:`~emukit.core.loop.loop_state.LoopState` as a real ``Y``. The model is updated once per **confirmed**
observation, never per ask.


In [ ]:
import numpy as np

from emukit.core import ContinuousParameter, ParameterSpace
from emukit.core.loop import FixedIntervalUpdater, SequentialPointCalculator, UserFunctionResult
from emukit.core.loop.loop_state import create_loop_state
from emukit.core.optimization import GradientAcquisitionOptimizer
from emukit.experimental_design.acquisitions import ModelVariance
from emukit.model_wrappers import SimpleGaussianProcessModel


## 1. Build a loop as usual

We use the dependency-free ``SimpleGaussianProcessModel`` here; the same ``ask`` / ``tell`` methods work with
any Emukit loop, including ``BayesianOptimizationLoop`` backed by GPy.


In [ ]:
space = ParameterSpace([ContinuousParameter("x", 0.0, 1.0)])

x_init = np.linspace(0.0, 1.0, 5)[:, None]
y_init = x_init ** 2

model = SimpleGaussianProcessModel(x_init, y_init)
candidate_point_calculator = SequentialPointCalculator(
    ModelVariance(model), GradientAcquisitionOptimizer(space)
)
model_updater = FixedIntervalUpdater(model)

loop_state = create_loop_state(x_init, y_init)

from emukit.core.loop import OuterLoop

loop = OuterLoop(candidate_point_calculator, model_updater, loop_state)


## 2. Claim candidate points with ``ask``

Every ``ask`` returns a dictionary containing the point ``X`` and a unique ``observation_id``. On a
continuous space we switch on the *penalize* strategy so pending points smoothly discourage the acquisition
from re-suggesting locations with an experiment already in flight. For discrete or categorical spaces the
default *exclude* strategy rejects exact duplicates instead.


In [ ]:
loop.enable_pending_point_handling(strategy="penalize", penalty_sigma=0.05)

claims = [loop.ask() for _ in range(3)]
for claim in claims:
    print(claim["observation_id"], claim["X"].flatten())


Claimed points are pending and have not produced any model update or new data in the loop state.

In [ ]:
print("Loop iteration:", loop.loop_state.iteration)
print("Data points in the loop state:", loop.loop_state.X.shape[0])
print("Pending observations:", [o.observation_id for o in loop.pending_tracker.pending])


## 3. Confirm results out of order with ``tell``

Suppose the third experiment finishes first. We report its result together with the id returned by ``ask``.
The result is validated against the originally claimed point before anything is stored.


In [ ]:
first_finished = claims[2]
result = UserFunctionResult(first_finished["X"][0], np.array([0.31]))
confirmed_ids = loop.tell(result, first_finished["observation_id"])
print("Confirmed:", confirmed_ids)
print("Loop iteration:", loop.loop_state.iteration)


### Repeated or conflicting reports are rejected

Once an id has reached a terminal status, reporting it again raises an error. The first confirmed
observation always wins, so a late duplicate response from an experiment queue cannot corrupt the data.


In [ ]:
duplicate = UserFunctionResult(first_finished["X"][0], np.array([0.99]))
try:
    loop.tell(duplicate, first_finished["observation_id"])
except ValueError as error:
    print("Rejected:", error)


## 4. Failures and cancellations only release the placeholder

Experiments can fail or be cancelled. Neither outcome creates an observation or triggers a model update:
the pending placeholder is simply released so the point can be suggested again later.


In [ ]:
loop.tell_failure(claims[1]["observation_id"])
loop.tell_cancelled(claims[0]["observation_id"])

print("Loop iteration:", loop.loop_state.iteration)
print(
    "Statuses:",
    {o.observation_id: o.status.value for o in loop.pending_tracker.observations()},
)


## 5. Summary of the lifecycle

| Status      | Meaning                                        | In ``LoopState``? | Model update? |
|-------------|------------------------------------------------|-------------------|---------------|
| pending     | claimed via ``ask``, experiment still running  | no                | no            |
| succeeded   | first real success confirmed via ``tell``      | yes               | yes           |
| failed      | reported via ``tell_failure``                  | no                | no            |
| cancelled   | reported via ``tell_cancelled``                | no                | no            |

The number of model updates therefore equals the number of **confirmed** experiments, not the number of
``ask`` calls.
